# Local Window Neighborhood Binarization Benchmarking

## 1. Objective
Before feeding document canvases into an OCR engine, binarization (turning them into pure black and white) is mandatory. This notebook establishes a comparative benchmark comparing Gaussian local adaptive thresholding window behaviors to prevent spatial artifacts (character hollowization) at high scaling zooms.

---

## 2. The Core Mathematical Concept: Stroke Width vs. Block Size

Adaptive thresholding computes a local threshold $T(x,y)$ for every individual pixel based on a sliding square window of size $B \times B$ (Block Size) and a constant offset $C$:
$$T(x, y) = \mu(x, y) - C$$
Where $\mu(x,y)$ is the Gaussian weighted average of the neighborhood. The choice of $B$ relative to the physical **Stroke Width** ($W_s$) of the characters determines the structural binarization scenario:

###  Scenario A: Solid Character Preservation ($B > W_s$)
*   **The Math:** By setting the Block Size ($B = 11$) wider than the physical stroke width of the scaled text characters, the sliding neighborhood window is guaranteed to encompass parts of the white background surrounding the letters.
*   **The Result:** The computed local mean $\mu(x,y)$ remains high, successfully classifying the entire inner body of the letter strokes as dark ink. The characters remain **structurally solid**, which is optimal for OCR engines.

### Scenario B: Text Hollowization/Outlining ($B \le W_s$)
*   **The Math:** If the Block Size is shrunken ($B = 3$) to a size narrower than the character stroke, the sliding window falls entirely inside the dark body of a letter stroke.
*   **The Result:** The local mean $\mu(x,y)$ drops significantly, forcing the thresholding logic to falsely interpret the center of the stroke as "background". This creates a destructive **hollow outline/double-edge artifact**, which completely breaks character contouring and halts OCR recognition.

In [ ]:
import fitz  
import cv2
import numpy as np
import os

def run_thresholding_comparative_benchmark(pdf_path, output_folder="otsu_experiment_output"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"[+] Created output folder: {output_folder}")

    # Open the document and load the first page for testing
    doc = fitz.open(pdf_path)
    page = doc.load_page(0)
    
    # Render the full page canvas at a baseline resolution (6.0x zoom)
    zoom_factor = 6.0
    matrix = fitz.Matrix(zoom_factor, zoom_factor)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    
    # Convert raw pixmap samples into a Grayscale Numpy array
    img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.h, pix.w, pix.n))
    img_gray = cv2.cvtColor(img_data, cv2.COLOR_RGB2GRAY)
    
    print(f"[+] Base image rendered at {zoom_factor}x zoom. Dimensions: {pix.w}x{pix.h} pixels.")
    print("-" * 75)

    # =========================================================================
    # CODE FOR SCENARIO A: Solid Black Text (Block Size > Stroke Width)
    # =========================================================================
    # Block Size is set to 11, which is physically wider than the 6.0x text stroke.
    # The sliding window always sees the white background, keeping the text filled.
    block_size_a = 11
    solid_output = cv2.adaptiveThreshold(
        img_gray, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 
        block_size_a, 2
    )
    output_path_a = os.path.join(output_folder, "scenario_a_solid.png")
    cv2.imwrite(output_path_a, solid_output)
    print(f"[✓] Saved Scenario A (Solid Text | Block Size={block_size_a}) -> {output_path_a}")

    # =========================================================================
    # CODE FOR SCENARIO B: Hollow / Outlined Text (Block Size < Stroke Width)
    # =========================================================================
    # Block Size is shrunk to 3, making it narrower than the 6.0x text stroke.
    # The sliding window falls entirely inside the black stroke, causing inversion.
    block_size_b = 3
    hollow_output = cv2.adaptiveThreshold(
        img_gray, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 
        block_size_b, 2
    )
    output_path_b = os.path.join(output_folder, "scenario_b_hollow.png")
    cv2.imwrite(output_path_b, hollow_output)
    print(f"[✓] Saved Scenario B (Hollow Text | Block Size={block_size_b}) -> {output_path_b}")
    print("-" * 75)
    print("[+] Benchmark complete! Open the folder to visually compare both matrix behaviors.")

if __name__ == "__main__":
    PDF_FILE_PATH = r"كشف حساب الراجحي.pdf"
    run_thresholding_comparative_benchmark(PDF_FILE_PATH)